In [ ]:
import os
import sys
import asyncio

# Windows + psycopg async
if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(
        asyncio.WindowsSelectorEventLoopPolicy()
    )

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from dataclasses import replace

from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from agents.primary.agent import build_primary_graph
from infrastructure.postgres import create_pool
from memory.commit import MemoryCommitAdapter
from memory.embeddings import MemoryEmbeddingService
from memory.verifier import build_memory_verifier
from memory.worker import MemoryWorker
from repositories.long_term_memory import PostgresLongTermMemoryRepository
from repositories.result_store import ResultStoreRepository
from services.long_term_memory import MemoryService
from dotenv import load_dotenv
from settings import get_settings
from utils.tracing import with_trace_config
from services.summarize import should_summarize, select_messages_to_summarize

# MCP servers (hotel 8004, flight 8003, ...) must be running first.
# Build graph ONCE with memory_service — do not recreate pool / rebuild later
# (second pool causes "Task was destroyed but it is pending").
load_dotenv()
get_settings.cache_clear()
base_settings = get_settings()
trustmem_model = (
    base_settings.long_term_memory_langmem_model
    if not str(base_settings.long_term_memory_langmem_model)
    .lower()
    .startswith("heuristic")
    else "gemini-2.5-flash"
)
settings = replace(
    base_settings,
    long_term_memory_recall_enabled=True,
    long_term_memory_write_enabled=True,
    # Notebook: process memory_jobs ngay trong turn (không cần worker process riêng)
    long_term_memory_sync_finalize=True,
    long_term_memory_extractor="langmem",
    long_term_memory_verifier="trustmem",
    long_term_memory_trustmem_model=trustmem_model,
    long_term_memory_trustmem_timeout_seconds=90,
)
pool = create_pool(settings)
await pool.open(wait=True)
repo = ResultStoreRepository(pool)
memory_repo = PostgresLongTermMemoryRepository(pool)
embedding_service = MemoryEmbeddingService(settings=settings)
memory_worker = MemoryWorker(
    pool=pool,
    settings=settings,
    repository=memory_repo,
    commit_adapter=MemoryCommitAdapter(
        repository=memory_repo,
        verifier=build_memory_verifier(settings),
        embedding_service=embedding_service,
    ),
)
memory_service = MemoryService(
    settings=settings,
    repository=memory_repo,
    processor=memory_worker,
    embedding_service=embedding_service,
)

checkpointer = AsyncPostgresSaver(pool)
await checkpointer.setup()
graph = await build_primary_graph(
    checkpointer=checkpointer,
    repo=repo,
    memory_service=memory_service,
)

# Use a fresh thread_id after message-history / summarize fixes so old
# corrupted checkpoints are not reused.
thread_id = "123"
user_id = "31082004"
config = with_trace_config(
    {"configurable": {"thread_id": thread_id, "user_id": user_id}},
    run_name="notebook_primary",
    tags=["notebook", "primary"],
    metadata={"thread_id": thread_id, "user_id": user_id},
)
print(
    "ready",
    {
        "thread_id": thread_id,
        "user_id": user_id,
        "sync_finalize": settings.long_term_memory_sync_finalize,
        "extractor": settings.long_term_memory_extractor,
        "verifier": settings.long_term_memory_verifier,
        "trustmem_model": settings.long_term_memory_trustmem_model,
        "trustmem_timeout": settings.long_term_memory_trustmem_timeout_seconds,
    },
)


In [ ]:

from IPython.display import Image, display

try:
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    pass

In [ ]:
_printed = set()

def _print_event(event: dict, _printed: set, max_length=5000):
    messages = event.get("messages")
    if not messages:
        return

    if not isinstance(messages, list):
        messages = [messages]

    for message in messages:
        message_id = getattr(message, "id", None)
        if message_id in _printed:
            continue

        msg_repr = message.pretty_repr(html=True)
        if len(msg_repr) > max_length:
            msg_repr = msg_repr[:max_length] + " ... (truncated)"
        print(msg_repr)
        if message_id:
            _printed.add(message_id)

In [ ]:
tutorial_questions = [
    "Tìm khách sạn ở Hà Nội, check-in 20/8/2026, check-out 23/8/2026.",
    "Cho tôi xem review của khách sạn thứ 3",
    # " Tìm vé một chiều Sài Gòn đi Hà Nội ngày 21/8/2026.",
    " Tôi muốn kiếm tour ở chỗ đó ngày 23/8/2026.",
]

_printed = set()

for question in tutorial_questions:
    events = graph.astream(
        {
            "messages": ("user", question),
            "user_id": user_id,
            "thread_id": thread_id,
        },
        config,
        stream_mode="values",
    )
    async for event in events:
        _print_event(event, _printed)

In [ ]:
import json
from pprint import pprint
from services.reference_resolver import resolve_item_reference

snapshot = await graph.aget_state(config)
values = snapshot.values
messages = values.get("messages") or []

print("=== META ===")
print("next:", snapshot.next)
print("user_id:", values.get("user_id"))
print("thread_id:", values.get("thread_id") or config["configurable"]["thread_id"])
print("dialog_state:", values.get("dialog_state"))
print("active_assistant:", values.get("active_assistant"))

print("\n=== SHORT-TERM MEMORY (trong checkpoint) ===")
print("summary:", values.get("summary"))
print("active_request_id:", values.get("active_request_id"))
print("latest_request_by_domain:")
pprint(values.get("latest_request_by_domain"))
print("requests:")
pprint(values.get("requests"))
print("request_results:")
pprint(values.get("request_results"))
print("visible_results:")
pprint(values.get("visible_results"))
print("selected_items:")
pprint(values.get("selected_items"))
print("flight_token_map keys:", list((values.get("flight_token_map") or {}).keys())[:10])

print("\n=== MESSAGES WINDOW ===")
print("message_count:", len(messages))
for i, msg in enumerate(messages):
    role = getattr(msg, "type", msg.__class__.__name__)
    content = getattr(msg, "content", "")
    preview = str(content).replace("\n", " ")[:160]
    tool_calls = getattr(msg, "tool_calls", None) or []
    extra = f" | tool_calls={[tc.get('name') for tc in tool_calls]}" if tool_calls else ""
    print(f"[{i}] {role}: {preview}{extra}")

print("\n=== RESOLVE 'thứ 2' BẰNG CODE ===")
resolved = resolve_item_reference(values, domain="hotel", position=2)
print(resolved)

# Nếu có search_id, tải payload tạm từ Result Store (không nằm trong State)
visible = values.get("visible_results") or {}
req_id = (values.get("latest_request_by_domain") or {}).get("hotel")
if req_id and req_id in visible:
    ref = visible[req_id]
    items = await repo.load_items(
        search_id=ref["search_id"],
        item_ids=ref.get("displayed_item_ids") or [],
        user_id=user_id,
        thread_id=thread_id,
    )
    print("\n=== RESULT STORE PAYLOAD (ngoài State) ===")
    print(json.dumps(items, ensure_ascii=False, indent=2)[:2000])


In [ ]:
tutorial_questions = [
    " Tìm khách sạn Đà Nẵng check in ngày 21/8/2026, check out ngày 22/8/2026.",
    " Tôi muốn kiếm tour ở Đà Nẵng ngày 22/8/2026.",
    " Cho tôi xem review của tour thứ 2",
]
user_id = "31082004"
thread_id = "444"
_printed = set()
config = with_trace_config(
    {"configurable": {"thread_id": thread_id, "user_id": user_id}},
    run_name="notebook_primary",
    tags=["notebook", "primary"],
    metadata={"thread_id": thread_id, "user_id": user_id},
)
for question in tutorial_questions:
    events = graph.astream(
        {"messages": ("user", question), "user_id": user_id, "thread_id": thread_id},
        config,
        stream_mode="values",
    )
    async for event in events:
        _print_event(event, _printed)
    snap = await graph.aget_state(config)
    msgs = snap.values.get("messages") or []
    old = select_messages_to_summarize(msgs)  # default keep_count=2
    print(
        f"\n>> {question.strip()!r}\n"
        f"   msgs={len(msgs)}  summary_len={len(snap.values.get('summary') or '')}\n"
        f"   would_remove={len(old)}  route={should_summarize(snap.values)}\n"
    )

In [ ]:
import json
from pprint import pprint
from services.reference_resolver import resolve_item_reference

snapshot = await graph.aget_state(config)
values = snapshot.values
messages = values.get("messages") or []

print("=== META ===")
print("next:", snapshot.next)
print("user_id:", values.get("user_id"))
print("thread_id:", values.get("thread_id") or config["configurable"]["thread_id"])
print("dialog_state:", values.get("dialog_state"))
print("active_assistant:", values.get("active_assistant"))

print("\n=== SHORT-TERM MEMORY (trong checkpoint) ===")
print("summary:", values.get("summary"))
print("active_request_id:", values.get("active_request_id"))
print("latest_request_by_domain:")
pprint(values.get("latest_request_by_domain"))
print("requests:")
pprint(values.get("requests"))
print("request_results:")
pprint(values.get("request_results"))
print("visible_results:")
pprint(values.get("visible_results"))
print("selected_items:")
pprint(values.get("selected_items"))
print("flight_token_map keys:", list((values.get("flight_token_map") or {}).keys())[:10])

print("\n=== MESSAGES WINDOW ===")
print("message_count:", len(messages))
for i, msg in enumerate(messages):
    role = getattr(msg, "type", msg.__class__.__name__)
    content = getattr(msg, "content", "")
    preview = str(content).replace("\n", " ")[:160]
    tool_calls = getattr(msg, "tool_calls", None) or []
    extra = f" | tool_calls={[tc.get('name') for tc in tool_calls]}" if tool_calls else ""
    print(f"[{i}] {role}: {preview}{extra}")

print("\n=== RESOLVE 'thứ 2' BẰNG CODE ===")
resolved = resolve_item_reference(values, domain="hotel", position=2)
print(resolved)

# Nếu có search_id, tải payload tạm từ Result Store (không nằm trong State)
visible = values.get("visible_results") or {}
req_id = (values.get("latest_request_by_domain") or {}).get("hotel")
if req_id and req_id in visible:
    ref = visible[req_id]
    items = await repo.load_items(
        search_id=ref["search_id"],
        item_ids=ref.get("displayed_item_ids") or [],
        user_id=user_id,
        thread_id=thread_id,
    )
    print("\n=== RESULT STORE PAYLOAD (ngoài State) ===")
    print(json.dumps(items, ensure_ascii=False, indent=2)[:2000])


In [ ]:
tutorial_questions = [
    "Tôi muốn kiếm chuyen bay từ tp hcm đến Phú Yên ngày 21/8/2026.",
    "Tôi muốn book chuyến thứ 2."
]
user_id = "31082004"
thread_id = "3334455"
_printed = set()
config = with_trace_config(
    {"configurable": {"thread_id": thread_id, "user_id": user_id}},
    run_name="notebook_primary",
    tags=["notebook", "primary"],
    metadata={"thread_id": thread_id, "user_id": user_id},
)
for question in tutorial_questions:
    events = graph.astream(
        {"messages": ("user", question), "user_id": user_id, "thread_id": thread_id},
        config,
        stream_mode="values",
    )
    async for event in events:
        _print_event(event, _printed)
    snap = await graph.aget_state(config)
    msgs = snap.values.get("messages") or []
    old = select_messages_to_summarize(msgs)  # default keep_count=2
    print(
        f"\n>> {question.strip()!r}\n"
        f"   msgs={len(msgs)}  summary_len={len(snap.values.get('summary') or '')}\n"
        f"   would_remove={len(old)}  route={should_summarize(snap.values)}\n"
    )

## TEST LONG TERM MEMORY

Chạy **Restart Kernel** → cell setup đầu notebook (một lần) → cell E2E ngay dưới đây.

- Không tạo thêm `create_pool` / không `build_primary_graph` lần 2 (gây pool pending + MCP `ConnectTimeout`).
- MCP hotel/flight phải đang chạy trước khi chạy cell setup.


In [ ]:
# Probe TrustMem LLM verifier (không cần MCP / graph turn).
# Chạy lại cell setup trước. Dấu hiệu LLM thật: model != heuristic-*, có 3 dimension scores, fallback_reason is None.

import json

from memory.consolidation import MemoryTransition, TransitionAction
from memory.long_term import MemoryCategory, MemoryDomain, TravelMemory
from memory.verifier import MemoryVerifierContext, build_memory_verifier
from psycopg.rows import dict_row

assert "settings" in dir() and "pool" in dir(), "Chạy cell setup đầu notebook trước."

verifier = build_memory_verifier(settings)
print("verifier_class:", type(verifier).__name__)
print("mode_setting:", settings.long_term_memory_verifier)
print("trustmem_model:", settings.long_term_memory_trustmem_model)
print("scorer:", getattr(verifier._scorer, "__name__", verifier._scorer))

candidate = TravelMemory(
    user_id=user_id,
    memory_text="ưu tiên khách sạn yên tĩnh, không gần biển khi đi công tác",
    category=MemoryCategory.HOTEL_PREFERENCE,
    domain=MemoryDomain.HOTEL,
    condition="công tác",
    evidence_text="Tôi muốn khách sạn yên tĩnh, tránh gần biển khi đi công tác",
    source_thread_id=thread_id,
)
chunk = [
    {
        "type": "human",
        "content": "Tôi muốn khách sạn yên tĩnh, tránh gần biển khi đi công tác",
    }
]
result = await verifier.evaluate(
    MemoryTransition(TransitionAction.INSERT, candidate=candidate),
    MemoryVerifierContext(chunk=chunk, old_memories=[], new_memories=[candidate]),
)
print("\n=== live evaluate ===")
print(json.dumps(result.to_dict(), ensure_ascii=False, indent=2, default=str))
if result.decision == "retry" or result.fallback_reason or not result.dimensions:
    print(
        "⚠️ TrustMem LLM failed —",
        result.fallback_reason or result.reasons,
    )
elif str(result.model).lower().startswith("heuristic"):
    print("⚠️ Đang heuristic, chưa gọi LLM. Set long_term_memory_trustmem_model = gemini-...")
else:
    print("✅ TrustMem LLM scorer chạy xong (mode=%s, model=%s)" % (result.mode, result.model))

print("\n=== latest memory_audit_records ===")
async with pool.connection() as conn:
    async with conn.cursor(row_factory=dict_row) as cur:
        await cur.execute(
            """
            SELECT created_at, decision, verifier_result
            FROM memory_audit_records
            WHERE user_id = %s
            ORDER BY created_at DESC
            LIMIT 5
            """,
            (user_id,),
        )
        rows = await cur.fetchall()
if not rows:
    print("(chưa có audit — chạy 1 turn chat có preference rồi chạy lại cell này)")
else:
    for row in rows:
        vr = row["verifier_result"] or {}
        dims = vr.get("dimensions") or {}
        print(
            f"{row['created_at']}  decision={row['decision']}"
            f"  mode={vr.get('mode')}  model={vr.get('model')}"
            f"  fallback={vr.get('fallback_reason')}"
        )
        for name, payload in dims.items():
            print(
                f"   {name}: score={payload.get('score')} "
                f"passed={payload.get('passed')} reason={payload.get('reason')}"
            )


In [ ]:
# Full TrustMem write path: extract → transition → verify_and_commit → audit.
# Chạy cell setup trước. Cell này ghi DB với user_id riêng (không đụng user notebook).

import json
import uuid

from memory.consolidation import calculate_transition
from memory.long_term import MemoryCategory, MemoryDomain, MemoryFamily, TravelMemory
from memory.verifier import MemoryVerifierContext, project_memory_state
from repositories.long_term_memory import MemorySearchFilters
from psycopg.rows import dict_row

assert "memory_worker" in dir() and "memory_repo" in dir(), "Chạy cell setup đầu notebook trước."

run_user = f"trustmem-e2e-{uuid.uuid4().hex[:8]}"
run_thread = f"thread-{uuid.uuid4().hex[:8]}"
user_text = "Tôi muốn khách sạn yên tĩnh, tránh gần biển khi đi công tác"
messages = [
    {"type": "human", "content": user_text},
    {
        "type": "ai",
        "content": "Mình đã ghi nhận: khi đi công tác bạn ưu tiên khách sạn yên tĩnh, không gần biển.",
    },
]

verifier = memory_worker._commit_adapter._verifier
print("=== 1) setup ===")
print(
    {
        "run_user": run_user,
        "run_thread": run_thread,
        "extractor": settings.long_term_memory_extractor,
        "verifier": settings.long_term_memory_verifier,
        "trustmem_model": settings.long_term_memory_trustmem_model,
        "verifier_class": type(verifier).__name__,
        "scorer": getattr(getattr(verifier, "_scorer", None), "__name__", None),
    }
)

existing = await memory_repo.search_active_memories(
    MemorySearchFilters(
        user_id=run_user,
        families=tuple(MemoryFamily),
        query=None,
        limit=settings.long_term_memory_text_search_limit,
    )
)
print("\n=== 2) M_old ===")
print(f"n={len(existing)}")
for mem in existing:
    print(f"  - [{mem.memory_id}] {mem.memory_text}")

print("\n=== 3) extractor ===")
candidates = await memory_worker._extractor.extract(
    messages,
    user_id=run_user,
    thread_id=run_thread,
    existing_active=existing,
)
if not candidates:
    print("LangMem không ra candidate → fallback fixture để vẫn chạy verify_and_commit")
    candidates = [
        TravelMemory(
            user_id=run_user,
            memory_text="ưu tiên khách sạn yên tĩnh, không gần biển khi đi công tác",
            category=MemoryCategory.HOTEL_PREFERENCE,
            domain=MemoryDomain.HOTEL,
            condition="công tác",
            evidence_text=user_text,
            source_thread_id=run_thread,
        )
    ]
for i, cand in enumerate(candidates, 1):
    print(f"  [{i}] {cand.category}: {cand.memory_text}")
    print(f"      evidence={cand.evidence_text!r}  condition={cand.condition!r}")

print("\n=== 4-5) transition + verify_and_commit ===")
for cand in candidates:
    transition = calculate_transition(cand, existing)
    projected = project_memory_state(transition, existing)
    print("\n--- transition ---")
    print(
        {
            "action": str(transition.action),
            "existing_memory_id": transition.existing_memory_id,
            "reasons": list(transition.reasons),
            "candidate": cand.memory_text,
        }
    )
    commit = await memory_worker._commit_adapter.verify_and_commit(
        transition=transition,
        user_id=run_user,
        thread_id=run_thread,
        job_id=None,
        verifier_context=MemoryVerifierContext(
            chunk=messages,
            old_memories=tuple(existing),
            new_memories=tuple(projected),
        ),
    )
    print("--- commit ---")
    print(
        {
            "decision": commit.decision,
            "affected_memory_ids": commit.affected_memory_ids,
            "reasons": commit.reasons,
        }
    )
    if commit.decision == "approve":
        existing = await memory_repo.search_active_memories(
            MemorySearchFilters(
                user_id=run_user,
                families=tuple(MemoryFamily),
                query=None,
                limit=settings.long_term_memory_text_search_limit,
            )
        )

print("\n=== 6) memory_audit_records (run_user) ===")
async with pool.connection() as conn:
    async with conn.cursor(row_factory=dict_row) as cur:
        await cur.execute(
            """
            SELECT created_at, decision, job_id, affected_memory_ids,
                   proposed_transition, rule_result, verifier_result
            FROM memory_audit_records
            WHERE user_id = %s
            ORDER BY created_at DESC
            LIMIT 5
            """,
            (run_user,),
        )
        rows = await cur.fetchall()
if not rows:
    print("(không có audit — verify_and_commit chưa ghi được)")
else:
    for row in rows:
        print(json.dumps(dict(row), ensure_ascii=False, indent=2, default=str))
        vr = row["verifier_result"] or {}
        if vr.get("mode") == "trustmem" and vr.get("dimensions") and not vr.get("fallback_reason"):
            print("✅ audit có TrustMem LLM (mode=trustmem, có dimension scores)")
        else:
            print(
                "⚠️ audit chưa phải TrustMem LLM đầy đủ:",
                {k: vr.get(k) for k in ("mode", "model", "fallback_reason")},
            )

print("\n=== 7) active memories after commit ===")
stored = await memory_repo.search_active_memories(
    MemorySearchFilters(
        user_id=run_user,
        families=tuple(MemoryFamily),
        query=None,
        limit=20,
    )
)
if not stored:
    print("(empty — commit không approve/insert)")
for mem in stored:
    print(f"  - [{mem.memory_id}] {mem.category}: {mem.memory_text}")


In [ ]:
# E2E: Primary + Long-Term Memory (hotel preference filtering)
# Prerequisites:
#   1) Run setup cell (cell 0) once — graph already has memory_service.
#   2) MCP hotel/flight servers are running.
# Do NOT create a second pool or call build_primary_graph again here.

import uuid
import json
from memory.long_term import TravelMemory, MemoryCategory, MemoryDomain

assert "graph" in dir() and "memory_repo" in dir() and "memory_service" in dir(), (
    "Chạy lại cell setup đầu notebook trước (cần graph + memory_repo + memory_service)."
)

print("recall_enabled:", settings.long_term_memory_recall_enabled)

user_id = f"test-mem-{uuid.uuid4().hex[:6]}"
thread_id = f"thread-{uuid.uuid4().hex[:6]}"

memory_text = (
    "Tôi ghét biển và không thích những nơi ồn ào. "
    "Tôi chỉ thích ở trung tâm thành phố."
)
mem = TravelMemory(
    user_id=user_id,
    memory_text=memory_text,
    category=MemoryCategory.HOTEL_PREFERENCE,
    domain=MemoryDomain.HOTEL,
    evidence_text="Tôi ghét biển, cho tôi ở trung tâm",
    source_thread_id="past-thread-123",
)
memory_id = await memory_repo.insert_memory(mem)
print(f"✅ Injected memory {memory_id} for {user_id}")
print(f"   text: {memory_text}")

# Sanity: recall API alone
recall = await memory_service.recall(user_id=user_id, query="khách sạn Nha Trang")
print(f"🔎 Direct recall: n={len(recall.recalled_memory_ids)}")
print(f"   context preview: {(recall.memory_context or '')[:200]!r}")

config = with_trace_config(
    {"configurable": {"thread_id": thread_id, "user_id": user_id}},
    run_name="notebook_primary_memory_test",
    tags=["notebook", "primary", "memory_test"],
    metadata={"thread_id": thread_id, "user_id": user_id},
)

_printed = set()
question = "Tìm khách sạn ở Nha Trang check in ngày 21/8/2026, check out ngày 22/8/2026."
print(f"\n👤 {question}\n" + "-" * 50)

events = graph.astream(
    {"messages": [("user", question)], "user_id": user_id, "thread_id": thread_id},
    config,
    stream_mode="values",
)
async for event in events:
    _print_event(event, _printed)

snap = await graph.aget_state(config)
state = snap.values
memory_context = state.get("memory_context") or ""
recalled_ids = state.get("recalled_memory_ids") or []

print("-" * 50)
print("\n🔍 ASSERTIONS:")
if memory_context and recalled_ids:
    print(f"  ✅ memory_recall OK — {len(recalled_ids)} id(s)")
    print(f"  🧠 {memory_context.strip()[:400]}")
else:
    print("  ❌ missing memory_context / recalled_memory_ids")
    print("     → check LONG_TERM_MEMORY_RECALL_ENABLED / settings override in cell 0")

tool_calls = [
    tc
    for m in state.get("messages") or []
    for tc in (getattr(m, "tool_calls", None) or [])
]
if tool_calls:
    args = tool_calls[0].get("args", {})
    print(f"  🛠️ first tool args: {json.dumps(args, ensure_ascii=False)}")

# Check hotel AI display filtering (ToolMessage / final printed hotels)
hotel_texts = []
for m in state.get("messages") or []:
    content = getattr(m, "content", "") or ""
    if isinstance(content, str) and (
        "accessibilityLabel" in content or "Giáp biển" in content or "trung tâm" in content
    ):
        hotel_texts.append(content)
joined = "\n".join(hotel_texts).lower()
if memory_context and recalled_ids and joined:
    beach_hits = joined.count("giáp biển") + joined.count("giap bien")
    if beach_hits == 0:
        print("  ✅ Hotel AI filtered display by memory (no 'Giáp biển' in printed hotels)")
    else:
        print(
            f"  ⚠️ Still printed beachfront ({beach_hits}× 'Giáp biển') — "
            "hotel AI should omit these when memory avoids beach"
        )
elif memory_context and recalled_ids:
    print("  ℹ️ No hotel print text found to verify preference filtering")
else:
    print("  ℹ️ Skip display-filter check (no recalled memory)")


In [ ]:
from memory.long_term import MemoryFamily
from repositories.long_term_memory import MemorySearchFilters
import uuid

tutorial_questions = [
    "Tôi Tôi ghét biển và không thích những nơi ồn ào.",
    "Tôi thích ở trung tâm thành phố.",
    "Tìm khách sạn ở Nha Trang check in ngày 21/8/2026, check out ngày 22/8/2026."
]
_run = uuid.uuid4().hex[:8]
user_id = f"ltm_01_{_run}"
thread_id = f"ltm_01_thread_{_run}"
print(f"user_id={user_id}  thread_id={thread_id}  extractor={settings.long_term_memory_extractor}")
_printed = set()
config = with_trace_config(
    {"configurable": {"thread_id": thread_id, "user_id": user_id}},
    run_name="notebook_primary",
    tags=["notebook", "primary"],
    metadata={"thread_id": thread_id, "user_id": user_id},
)
for question in tutorial_questions:
    events = graph.astream(
        {"messages": ("user", question), "user_id": user_id, "thread_id": thread_id},
        config,
        stream_mode="values",
    )
    async for event in events:
        _print_event(event, _printed)
    snap = await graph.aget_state(config)
    vals = snap.values
    msgs = vals.get("messages") or []
    old = select_messages_to_summarize(msgs)  # default keep_count=2

    memory_context = (vals.get("memory_context") or "").strip()
    recalled_ids = vals.get("recalled_memory_ids") or []
    memory_job_id = vals.get("memory_job_id")

    # Stored active memories for this user (each segment)
    stored = await memory_repo.search_active_memories(
        MemorySearchFilters(
            user_id=user_id,
            families=tuple(MemoryFamily),
            query=None,
            limit=50,
        )
    )

    print(
        f"\n>> {question.strip()!r}\n"
        f"   msgs={len(msgs)}  summary_len={len(vals.get('summary') or '')}\n"
        f"   would_remove={len(old)}  route={should_summarize(vals)}\n"
        f"   memory_job_id={memory_job_id}\n"
        f"   recalled_ids={recalled_ids}\n"
        f"   memory_context:\n{memory_context or '   (empty)'}\n"
        f"   stored_memories ({len(stored)}):"
    )
    if not stored:
        print("   (none)")
    else:
        for m in stored:
            mid = (m.memory_id or "")[:8]
            print(f"   - [{mid}] {m.category}: {m.memory_text}")


In [ ]:
from memory.long_term import MemoryFamily
from repositories.long_term_memory import MemorySearchFilters
import uuid

async def _run_turn(question: str, *, user_id: str, thread_id: str, _printed: set):
    config = with_trace_config(
        {"configurable": {"thread_id": thread_id, "user_id": user_id}},
        run_name="notebook_primary_ltm_cases",
        tags=["notebook", "primary", "ltm_cases"],
        metadata={"thread_id": thread_id, "user_id": user_id},
    )
    events = graph.astream(
        {"messages": ("user", question), "user_id": user_id, "thread_id": thread_id},
        config,
        stream_mode="values",
    )
    async for event in events:
        _print_event(event, _printed)

    snap = await graph.aget_state(config)
    vals = snap.values
    msgs = vals.get("messages") or []
    old = select_messages_to_summarize(msgs)
    memory_context = (vals.get("memory_context") or "").strip()
    recalled_ids = vals.get("recalled_memory_ids") or []
    memory_job_id = vals.get("memory_job_id")
    stored = await memory_repo.search_active_memories(
        MemorySearchFilters(
            user_id=user_id,
            families=tuple(MemoryFamily),
            query=None,
            limit=50,
        )
    )
    print(
        f"\n>> {question.strip()!r}\n"
        f"   msgs={len(msgs)}  summary_len={len(vals.get('summary') or '')}\n"
        f"   would_remove={len(old)}  route={should_summarize(vals)}\n"
        f"   memory_job_id={memory_job_id}\n"
        f"   recalled_ids={recalled_ids}\n"
        f"   memory_context:\n{memory_context or '   (empty)'}\n"
        f"   stored_memories ({len(stored)}):"
    )
    if not stored:
        print("   (none)")
    else:
        for m in stored:
            mid = (m.memory_id or "")[:8]
            print(f"   - [{mid}] {m.category}: {m.memory_text}")
    return vals, stored


async def run_ltm_primary_cases():
    """E2E cases qua primary graph + sync_finalize.
    Mỗi lần chạy sinh run_id mới → user/thread mới (tránh dính checkpoint/memory cũ).
    """
    run_id = uuid.uuid4().hex[:8]
    print(f"run_id={run_id}  extractor={settings.long_term_memory_extractor}")

    cases = [
        {
            "name": "01_hotel_pref_then_search",
            "expect": "Lưu ghét biển + thích trung tâm; search Nha Trang không in Giáp biển",
            "questions": [
                "Tôi ghét biển và không thích những nơi ồn ào.",
                "Tôi thích ở trung tâm thành phố.",
                "Tìm khách sạn ở Nha Trang check in ngày 21/8/2026, check out ngày 22/8/2026.",
            ],
        },
        {
            "name": "02_recall_only_search",
            "expect": "User mới không có memory → search bình thường, recalled rỗng",
            "questions": [
                "Tìm khách sạn ở Nha Trang check in ngày 21/8/2026, check out ngày 22/8/2026.",
            ],
        },
        {
            "name": "03_conflict_supersede",
            "expect": "Thích gần biển → đổi thành không thích biển / thích trung tâm (supersede)",
            "questions": [
                "Tôi thích khách sạn gần biển.",
                "Thôi tôi không thích gần biển nữa, tôi thích ở trung tâm thành phố.",
                "Tìm khách sạn ở Đà Nẵng check in 25/8/2026, check out 26/8/2026.",
            ],
        },
        {
            "name": "04_flight_pref",
            "expect": "Lưu flight preference (bay thẳng); search flight áp dụng nếu có",
            "questions": [
                "Tôi ưu tiên bay thẳng, không thích chuyến nối.",
                "Tìm vé máy bay từ Sài Gòn đi Hà Nội ngày 21/8/2026 cho 1 người lớn.",
            ],
        },
        {
            "name": "05_non_durable_chitchat",
            "expect": "Chào hỏi / trời đẹp → không lưu durable memory (job skipped hoặc stored=0)",
            "questions": [
                "Xin chào",
                "Hôm nay trời đẹp quá",
            ],
        },
        {
            "name": "06_multi_pref_hotel_budget",
            "expect": "Nhiều pref: budget + không hồ bơi; search hotel nhớ cả hai",
            "questions": [
                "Ngân sách khách sạn của tôi khoảng dưới 800 nghìn một đêm.",
                "Tôi không cần hồ bơi.",
                "Tìm khách sạn ở Nha Trang check in 21/8/2026, check out 22/8/2026.",
            ],
        },
        {
            "name": "07_profile_fact_kids",
            "expect": "Profile: đi cùng trẻ em → search hotel truyền children_age",
            "questions": [
                "Tôi thường đi du lịch cùng 1 trẻ em 8 tuổi.",
                "Tìm khách sạn ở Hội An check in 28/8/2026, check out 29/8/2026 cho 2 người lớn và 1 trẻ 8 tuổi.",
            ],
        },
        {
            "name": "08_car_pref",
            "expect": "Lưu car preference (số tự động / tự lái)",
            "questions": [
                "Tôi thích thuê xe tự lái, xe số tự động.",
                "Tôi cần thuê xe ở Đà Nẵng từ 21/8/2026 đến 23/8/2026.",
            ],
        },
    ]

    for case in cases:
        uid = f"ltm_{case['name']}_{run_id}"
        tid = f"thread_{case['name']}_{run_id}"
        _printed = set()
        print("\n" + "=" * 72)
        print(f"CASE {case['name']}")
        print(f"expect: {case['expect']}")
        print(f"user_id={uid}  thread_id={tid}")
        print("=" * 72)
        for q in case["questions"]:
            await _run_turn(q, user_id=uid, thread_id=tid, _printed=_printed)


await run_ltm_primary_cases()


In [ ]:
from memory.long_term import MemoryFamily
from repositories.long_term_memory import MemorySearchFilters

async def _run_turn(question: str, *, user_id: str, thread_id: str, _printed: set):
    config = with_trace_config(
        {"configurable": {"thread_id": thread_id, "user_id": user_id}},
        run_name="notebook_primary_ltm_cases",
        tags=["notebook", "primary", "ltm_cases"],
        metadata={"thread_id": thread_id, "user_id": user_id},
    )
    events = graph.astream(
        {"messages": ("user", question), "user_id": user_id, "thread_id": thread_id},
        config,
        stream_mode="values",
    )
    async for event in events:
        _print_event(event, _printed)

    snap = await graph.aget_state(config)
    vals = snap.values
    msgs = vals.get("messages") or []
    old = select_messages_to_summarize(msgs)
    memory_context = (vals.get("memory_context") or "").strip()
    recalled_ids = vals.get("recalled_memory_ids") or []
    memory_job_id = vals.get("memory_job_id")
    stored = await memory_repo.search_active_memories(
        MemorySearchFilters(
            user_id=user_id,
            families=tuple(MemoryFamily),
            query=None,
            limit=50,
        )
    )
    print(
        f"\n>> {question.strip()!r}\n"
        f"   msgs={len(msgs)}  summary_len={len(vals.get('summary') or '')}\n"
        f"   would_remove={len(old)}  route={should_summarize(vals)}\n"
        f"   memory_job_id={memory_job_id}\n"
        f"   recalled_ids={recalled_ids}\n"
        f"   memory_context:\n{memory_context or '   (empty)'}\n"
        f"   stored_memories ({len(stored)}):"
    )
    if not stored:
        print("   (none)")
    else:
        for m in stored:
            mid = (m.memory_id or "")[:8]
            print(f"   - [{mid}] {m.category}: {m.memory_text}")
    return vals, stored


# Subset runner — fresh ids mỗi lần (cùng pattern cell đầy đủ)
import uuid

async def run_ltm_primary_cases_subset():
    run_id = uuid.uuid4().hex[:8]
    print(f"run_id={run_id}  extractor={settings.long_term_memory_extractor}")
    cases = [
        {
            "name": "07_profile_fact_kids",
            "expect": "Profile: đi cùng trẻ em → search hotel truyền children_age",
            "questions": [
                "Tôi thường đi du lịch cùng 1 trẻ em 8 tuổi.",
                "Tìm khách sạn ở Hội An check in 28/8/2026, check out 29/8/2026 cho 2 người lớn và 1 trẻ 8 tuổi.",
            ],
        },
    ]
    for case in cases:
        uid = f"ltm_{case['name']}_{run_id}"
        tid = f"thread_{case['name']}_{run_id}"
        _printed = set()
        print("\n" + "=" * 72)
        print(f"CASE {case['name']}")
        print(f"expect: {case['expect']}")
        print(f"user_id={uid}  thread_id={tid}")
        print("=" * 72)
        for q in case["questions"]:
            await _run_turn(q, user_id=uid, thread_id=tid, _printed=_printed)


await run_ltm_primary_cases_subset()


In [ ]:
from memory.long_term import MemoryFamily
from repositories.long_term_memory import MemorySearchFilters

async def _run_turn(question: str, *, user_id: str, thread_id: str, _printed: set):
    config = with_trace_config(
        {"configurable": {"thread_id": thread_id, "user_id": user_id}},
        run_name="notebook_primary_ltm_cases",
        tags=["notebook", "primary", "ltm_cases"],
        metadata={"thread_id": thread_id, "user_id": user_id},
    )
    events = graph.astream(
        {"messages": ("user", question), "user_id": user_id, "thread_id": thread_id},
        config,
        stream_mode="values",
    )
    async for event in events:
        _print_event(event, _printed)

    snap = await graph.aget_state(config)
    vals = snap.values
    msgs = vals.get("messages") or []
    old = select_messages_to_summarize(msgs)
    memory_context = (vals.get("memory_context") or "").strip()
    recalled_ids = vals.get("recalled_memory_ids") or []
    memory_job_id = vals.get("memory_job_id")
    stored = await memory_repo.search_active_memories(
        MemorySearchFilters(
            user_id=user_id,
            families=tuple(MemoryFamily),
            query=None,
            limit=50,
        )
    )
    print(
        f"\n>> {question.strip()!r}\n"
        f"   msgs={len(msgs)}  summary_len={len(vals.get('summary') or '')}\n"
        f"   would_remove={len(old)}  route={should_summarize(vals)}\n"
        f"   memory_job_id={memory_job_id}\n"
        f"   recalled_ids={recalled_ids}\n"
        f"   memory_context:\n{memory_context or '   (empty)'}\n"
        f"   stored_memories ({len(stored)}):"
    )
    if not stored:
        print("   (none)")
    else:
        for m in stored:
            mid = (m.memory_id or "")[:8]
            print(f"   - [{mid}] {m.category}: {m.memory_text}")
    return vals, stored


# Flight-only subset — fresh ids mỗi lần
import uuid

async def run_ltm_flight_case():
    run_id = uuid.uuid4().hex[:8]
    print(f"run_id={run_id}  extractor={settings.long_term_memory_extractor}")
    cases = [
        {
            "name": "04_flight_pref",
            "expect": "Lưu flight preference (bay thẳng); search flight áp dụng nếu có",
            "questions": [
                "Tôi thích chuyến bay chiều  cho đỡ mệt, bay thẳng.",
                "Tìm vé máy bay từ Sài Gòn đi Tuy Hòa ngày 21/8/2026 cho 1 người lớn.",
            ],
        }
    ]
    for case in cases:
        uid = f"ltm_{case['name']}_{run_id}"
        tid = f"thread_{case['name']}_{run_id}"
        _printed = set()
        print("\n" + "=" * 72)
        print(f"CASE {case['name']}")
        print(f"expect: {case['expect']}")
        print(f"user_id={uid}  thread_id={tid}")
        print("=" * 72)
        for q in case["questions"]:
            await _run_turn(q, user_id=uid, thread_id=tid, _printed=_printed)


await run_ltm_flight_case()


In [ ]:
from memory.long_term import MemoryFamily
from repositories.long_term_memory import MemorySearchFilters

async def _run_turn(question: str, *, user_id: str, thread_id: str, _printed: set):
    config = with_trace_config(
        {"configurable": {"thread_id": thread_id, "user_id": user_id}},
        run_name="notebook_primary_ltm_cases",
        tags=["notebook", "primary", "ltm_cases"],
        metadata={"thread_id": thread_id, "user_id": user_id},
    )
    events = graph.astream(
        {"messages": ("user", question), "user_id": user_id, "thread_id": thread_id},
        config,
        stream_mode="values",
    )
    async for event in events:
        _print_event(event, _printed)

    snap = await graph.aget_state(config)
    vals = snap.values
    msgs = vals.get("messages") or []
    old = select_messages_to_summarize(msgs)
    memory_context = (vals.get("memory_context") or "").strip()
    recalled_ids = vals.get("recalled_memory_ids") or []
    memory_job_id = vals.get("memory_job_id")
    stored = await memory_repo.search_active_memories(
        MemorySearchFilters(
            user_id=user_id,
            families=tuple(MemoryFamily),
            query=None,
            limit=50,
        )
    )
    print(
        f"\n>> {question.strip()!r}\n"
        f"   msgs={len(msgs)}  summary_len={len(vals.get('summary') or '')}\n"
        f"   would_remove={len(old)}  route={should_summarize(vals)}\n"
        f"   memory_job_id={memory_job_id}\n"
        f"   recalled_ids={recalled_ids}\n"
        f"   memory_context:\n{memory_context or '   (empty)'}\n"
        f"   stored_memories ({len(stored)}):"
    )
    if not stored:
        print("   (none)")
    else:
        for m in stored:
            mid = (m.memory_id or "")[:8]
            print(f"   - [{mid}] {m.category}: {m.memory_text}")
    return vals, stored


# Flight-only subset — fresh ids mỗi lần
import uuid

async def run_ltm_flight_case():
    run_id = uuid.uuid4().hex[:8]
    print(f"run_id={run_id}  extractor={settings.long_term_memory_extractor}")
    cases = [
        {
            "name": "04_flight_pref",
            "expect": "Lưu flight preference (bay thẳng); search flight áp dụng nếu có",
            "questions": [
                "Tôi thích chuyến bay tầm đâu đó sáng cho đỡ mệt, bay thẳng.",
                "Tìm vé máy bay từ Sài Gòn đi Tuy Hòa ngày 21/8/2026 cho 1 người lớn.",
            ],
        }
    ]
    for case in cases:
        uid = f"ltm_{case['name']}_{run_id}"
        tid = f"thread_{case['name']}_{run_id}"
        _printed = set()
        print("\n" + "=" * 72)
        print(f"CASE {case['name']}")
        print(f"expect: {case['expect']}")
        print(f"user_id={uid}  thread_id={tid}")
        print("=" * 72)
        for q in case["questions"]:
            await _run_turn(q, user_id=uid, thread_id=tid, _printed=_printed)


await run_ltm_flight_case()


In [ ]:
# PGVector-only subset — fresh ids mỗi lần
# Requires migration 0006 and GOOGLE_API_KEY to be configured.
from dataclasses import replace
import uuid

from memory.embeddings import memory_content_hash
from memory.long_term import (
    MemoryCategory,
    MemoryDomain,
    MemoryFamily,
    TravelMemory,
)
from repositories.long_term_memory import MemoryEmbeddingRecord, MemorySearchFilters
from services.long_term_memory import MemoryService


async def run_pgvector_case():
    run_id = uuid.uuid4().hex[:8]
    print(f"run_id={run_id}  model={embedding_service.model}")
    cases = [
        {
            "name": "01_semantic_hotel_pref",
            "expect": "Câu hỏi diễn đạt khác từ khóa vẫn recall đúng hotel preference bằng pgvector",
            "query": "Cần nơi lưu trú ít tiếng ồn, thuận tiện đi bộ đến các điểm tham quan.",
            "memories": [
                {
                    "memory_text": "Tôi ưu tiên khách sạn yên tĩnh ở trung tâm thành phố.",
                    "category": MemoryCategory.HOTEL_PREFERENCE,
                    "domain": MemoryDomain.HOTEL,
                    "evidence_text": "Người dùng muốn khách sạn yên tĩnh ở trung tâm.",
                },
                {
                    "memory_text": "Tôi chỉ chọn chuyến bay thẳng và không muốn nối chuyến.",
                    "category": MemoryCategory.FLIGHT_PREFERENCE,
                    "domain": MemoryDomain.FLIGHT,
                    "evidence_text": "Người dùng ưu tiên chuyến bay thẳng.",
                },
            ],
        }
    ]

    for case in cases:
        uid = f"pgvector_{case['name']}_{run_id}"
        tid = f"thread_{case['name']}_{run_id}"
        inserted_ids = []
        print("\n" + "=" * 72)
        print(f"CASE {case['name']}")
        print(f"expect: {case['expect']}")
        print(f"user_id={uid}  thread_id={tid}")
        print("=" * 72)

        try:
            for item in case["memories"]:
                memory = TravelMemory(
                    user_id=uid,
                    source_thread_id=tid,
                    **item,
                )
                memory_id = await memory_repo.insert_memory(memory)
                inserted_ids.append(memory_id)
                vector = await embedding_service.embed_memory(memory)
                await memory_repo.upsert_memory_embedding(
                    MemoryEmbeddingRecord(
                        memory_id=memory_id,
                        embedding=vector,
                        embedding_model=embedding_service.model,
                        embedding_dims=embedding_service.dims,
                        content_hash=memory_content_hash(
                            memory,
                            model=embedding_service.model,
                        ),
                    )
                )

            filters = MemorySearchFilters(
                user_id=uid,
                families=(MemoryFamily.TRAVEL_PREFERENCES,),
                query=case["query"],
                limit=10,
            )
            lexical_hits = await memory_repo.search_active_memories(filters)
            assert lexical_hits == [], "Query unexpectedly matched lexical ILIKE search"

            query_vector = await embedding_service.embed_query(case["query"])
            vector_hits = await memory_repo.semantic_search_active_memories(
                filters,
                query_embedding=query_vector,
                embedding_model=embedding_service.model,
                embedding_dims=embedding_service.dims,
                distance_threshold=2.0,
            )
            assert vector_hits, "pgvector returned no semantic matches"
            assert vector_hits[0].memory_id == inserted_ids[0], (
                f"Expected hotel preference first, got: {vector_hits[0].memory_text}"
            )

            vector_settings = replace(
                settings,
                long_term_memory_vector_search_enabled=True,
                long_term_memory_vector_fallback_enabled=False,
                long_term_memory_vector_distance_threshold=2.0,
            )
            vector_service = MemoryService(
                settings=vector_settings,
                repository=memory_repo,
                embedding_service=embedding_service,
            )
            recall = await vector_service.recall(
                user_id=uid,
                query=case["query"],
                families=(MemoryFamily.TRAVEL_PREFERENCES,),
            )
            assert recall.recalled_memory_ids[0] == inserted_ids[0]

            print(f">> query: {case['query']!r}")
            print(f"   lexical_hits={len(lexical_hits)}")
            print(f"   recalled_ids={recall.recalled_memory_ids}")
            print("   vector ranking:")
            for rank, hit in enumerate(vector_hits, start=1):
                print(f"   {rank}. [{hit.memory_id[:8]}] {hit.memory_text}")
            print("✅ pgvector semantic search passed")
        finally:
            async with pool.connection() as conn:
                await conn.execute(
                    "DELETE FROM long_term_memories WHERE user_id = %(user_id)s",
                    {"user_id": uid},
                )


await run_pgvector_case()


In [ ]:
# Semantic duplicate case — khác từ nhưng cùng nghĩa
# Mục tiêu: LangMem nhìn existing memory và quyết định không insert bản trùng nghĩa.
import uuid

from memory.long_term import MemoryCategory, MemoryFamily
from repositories.long_term_memory import MemorySearchFilters


async def run_ltm_semantic_duplicate_case():
    run_id = uuid.uuid4().hex[:8]
    print(f"run_id={run_id}  extractor={settings.long_term_memory_extractor}")
    cases = [
        {
            "name": "09_semantic_duplicate_no_insert",
            "expect": "Hai câu khác từ nhưng cùng nghĩa → chỉ giữ 1 active flight preference",
            "questions": [
                "Khi đi máy bay, tôi luôn ưu tiên chuyến bay thẳng.",
                "Tôi chỉ muốn hành trình hàng không không phải đổi máy bay giữa chặng.",
            ],
        }
    ]

    for case in cases:
        uid = f"ltm_{case['name']}_{run_id}"
        tid = f"thread_{case['name']}_{run_id}"
        _printed = set()
        counts = []
        print("\n" + "=" * 72)
        print(f"CASE {case['name']}")
        print(f"expect: {case['expect']}")
        print(f"user_id={uid}  thread_id={tid}")
        print("=" * 72)

        for q in case["questions"]:
            await _run_turn(q, user_id=uid, thread_id=tid, _printed=_printed)
            active = await memory_repo.search_active_memories(
                MemorySearchFilters(
                    user_id=uid,
                    families=(MemoryFamily.TRAVEL_PREFERENCES,),
                    query=None,
                    limit=50,
                )
            )
            flight_memories = [
                memory
                for memory in active
                if memory.category == MemoryCategory.FLIGHT_PREFERENCE
            ]
            counts.append(len(flight_memories))
            print(f"   active flight memories after turn={len(flight_memories)}")

        async with pool.connection() as conn:
            audit_rows = await (
                await conn.execute(
                    """
                    SELECT decision, proposed_transition, created_at
                    FROM memory_audit_records
                    WHERE user_id = %(user_id)s
                    ORDER BY created_at ASC
                    """,
                    {"user_id": uid},
                )
            ).fetchall()

        print(f"\n   counts_by_turn={counts}")
        print("   audit decisions:")
        for row in audit_rows:
            transition = row["proposed_transition"] or {}
            print(
                f"   - decision={row['decision']} "
                f"action={transition.get('action')} "
                f"reasons={transition.get('reasons')}"
            )

        assert counts[0] == 1, "Turn đầu phải tạo đúng 1 flight preference"
        assert counts[-1] == 1, (
            "Semantic duplicate đã bị insert thành memory mới; "
            "LLM/consolidation cần quyết định NOOP hoặc không tạo candidate."
        )
        print("✅ Semantic duplicate was not inserted")


await run_ltm_semantic_duplicate_case()
